In [ ]:
#Import Libraries
# Step 1: Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,classification_report, confusion_matrix, roc_auc_score, roc_curve

# For handling imbalance
#!pip install imbalanced-learn
from imblearn.over_sampling import SMOTE


In [ ]:
#load the data
# Step 2: Load dataset
df = pd.read_csv("Fraud.csv")  # Replace with your actual file name

# Basic info
print("Shape of dataset:", df.shape)
print("\nFirst 5 rows:\n", df.head())

# Check if fraud cases exist
print("\nFraud count:\n", df['isFraud'].value_counts())


In [ ]:
#Basic Cleaning

# Step 3: Data Cleaning

# Check missing values
print("\nMissing values per column:\n", df.isnull().sum())

# Drop rows with missing values (if any)
df.dropna(inplace=True)

# Drop irrelevant columns if present
if 'isFlaggedFraud' in df.columns:
    df.drop('isFlaggedFraud', axis=1, inplace=True)

# Confirm shape after cleaning
print("\nShape after cleaning:", df.shape)

In [ ]:
#Encode Categorical Column
# Step 4: Encode 'type' column
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['type'] = le.fit_transform(df['type'].astype(str))

print("\nUnique values in 'type' after encoding:", df['type'].unique())


In [ ]:
#Split Data into Train and Test Sets
# Drop ID-like columns
drop_cols = ['nameOrig', 'nameDest']
for col in drop_cols:
    if col in df.columns:
        df.drop(col, axis=1, inplace=True)

# Define Features and Target again
X = df.drop(['isFraud'], axis=1)
y = df['isFraud']

# Split again
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)

In [ ]:
#Handle Class Imbalance with SMOTE
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print("Before SMOTE:", y_train.value_counts())
print("After SMOTE:", y_train_res.value_counts())

In [ ]:
#Train Random Forest Model
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_res, y_train_res)
print("Classes in model:", rf_model.classes_)


In [ ]:
y_pred = rf_model.predict(X_test)
y_prob = rf_model.predict_proba(X_test)[:, 1]

In [ ]:
print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nROC-AUC Score:", roc_auc_score(y_test, y_prob))


In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob)
plt.figure(figsize=(8,6))
plt.plot(fpr, tpr, label='Random Forest (AUC = %.4f)' % roc_auc_score(y_test, y_prob))
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc='lower right')
plt.show()